# HAEM5 Reporter — cell classifier training (Colab)

Trains the single-cell model used by the **HAEM5 Reporter** app's AI-assisted differential counter, and exports it as `cell_model.onnx` + `cell_labels.json` for loading into the app.

**Why a blast-containing dataset:** WHO-HAEM5 classification turns on blast percentages, so the model must see blasts. The default source is **AML-Cytomorphology_LMU** (Matek et al., *Nat Mach Intell* 2019; peripheral blood smears from AML patients and controls, expert-annotated, ~18,000 single-cell images, TCIA / Kaggle mirror). You can add the **Barcelona PBC** normal-cell set (or your Capstone B zip) to enrich normal classes, erythroblasts and platelets.

**What it outputs (app vocabulary):** blast · promyelocyte · myelocyte · metamyelocyte · band neutrophil · segmented neutrophil · eosinophil · basophil · monocyte · lymphocyte · atypical lymphocyte · erythroblast · platelet · smudge cell

**Steps**
1. *Runtime › Change runtime type › T4 GPU*.
2. *Runtime › Run all*. Cell 5 downloads AML-Cytomorphology_LMU from Kaggle automatically (if asked, upload your `kaggle.json` key).
3. Optional: add extra zips from Google Drive in cell 2 (e.g. PBC or your Capstone B zip).
4. **Check the sample grid in cell 6** — confirm every folder landed in the right class (you are the ground truth here).
5. At the end, `cell_model.onnx` and `cell_labels.json` download automatically → load both in the app (*Counter › AI-assisted*).

> Research/teaching prototype. Not a medical device; outputs require pathologist verification.

In [ ]:
# 1. Install only what is missing (Kaggle and Colab already include most packages)
import importlib.util, subprocess, sys, socket
need = [p for p, m in [("timm", "timm"), ("onnx", "onnx"), ("onnxruntime", "onnxruntime"), ("kagglehub", "kagglehub"),
                       ("scikit-learn", "sklearn"), ("seaborn", "seaborn")] if importlib.util.find_spec(m) is None]
def online():
    try: socket.create_connection(("pypi.org", 443), timeout=5).close(); return True
    except OSError: return False
if not online():
    raise SystemExit("❌ No internet. On Kaggle: right panel › Session options (or Settings) › Internet → ON "
                     "(needs a phone-verified account), then run this cell again.")
if need:
    print("Installing:", need)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])
print("✓ Packages ready")

In [ ]:
# 2. Configuration — edit zip names to match what you uploaded to Drive
# EASIEST (Colab): download the dataset straight from Kaggle inside Colab — no manual upload needed.
KAGGLE_DATASETS = ["walkersneps/aml-cytomorphology-lmu",            # blasts + AML-patient cells (Munich)
                   "orvile/microscopic-peripheral-blood-cell-images",  # Barcelona PBC: platelets, erythroblasts, IG stages
                   "andrewmvd/bone-marrow-cell-classification"]         # Munich bone marrow: plasma, hairy, immature lymphoid, proerythroblasts
USE_KAGGLE_DOWNLOAD = True

DATASETS = [
    # Optional extra zips already in your Google Drive (path, short name). Missing ones are skipped.
    # ("/content/drive/MyDrive/PBC_dataset_normal_DIB.zip", "PBC"),
    # ("/content/drive/MyDrive/Cappstone_project_B.zip", "CAPSTONE"),     # optional (no blasts)
]

# App vocabulary (order = model output order). Do not rename — the app relies on these keys.
CLASSES = ["blast", "promyelocyte", "myelocyte", "metamyelocyte", "band_neutrophil",
           "segmented_neutrophil", "eosinophil", "basophil", "monocyte", "lymphocyte",
           "atypical_lymphocyte", "erythroblast", "platelet", "smudge",
           "plasma_cell", "hairy_cell", "immature_lymphocyte", "proerythroblast", "artefact"]

# Folder-name (or filename-prefix) codes -> app class.  Matching is case-insensitive.
# AML-Cytomorphology_LMU codes (Matek 2019). VERIFY with the sample grid in cell 6:
LABEL_MAP = {
    "MYO": "blast",          # myeloblast
    "MOB": "blast",          # monoblast
    "PMO": "promyelocyte",
    "PMB": "promyelocyte",   # bilobed promyelocyte
    "MYB": "myelocyte",
    "MMZ": "metamyelocyte",
    "NGB": "band_neutrophil",
    "NGS": "segmented_neutrophil",
    "EOS": "eosinophil",
    "BAS": "basophil",
    "MON": "monocyte",
    "LYT": "lymphocyte",
    "LYA": "atypical_lymphocyte",
    "EBO": "erythroblast",
    "KSC": "smudge",
    # Munich bone-marrow codes (Matek 2021):
    "BLA": "blast", "PLM": "plasma_cell", "HAC": "hairy_cell", "LYI": "immature_lymphocyte",
    "PEB": "proerythroblast", "ART": "artefact",
    # Barcelona PBC / Capstone filename prefixes and folder names:
    "BA": "basophil", "EO": "eosinophil", "LY": "lymphocyte", "MO": "monocyte",
    "SNE": "segmented_neutrophil", "BNE": "band_neutrophil", "NEUTROPHIL": "segmented_neutrophil",
    "MY": "myelocyte", "MMY": "metamyelocyte", "PMY": "promyelocyte",
    "ERB": "erythroblast", "PLATELET": "platelet", "PLT": "platelet",
}
SKIP_CODES = {"IG", "NIF", "OTH", "ABE", "FGC"}   # pooled IG; not identifiable; other; too few abnormal eos / faggot cells

QUICK = True                 # True = tonight's fast run (~45 min). Set False later for the full run (adds bone-marrow set, 15 epochs).
TEST_FRAC = 0.15             # held out until the very end (cell 9)
VAL_FRAC  = 0.15             # used each epoch to pick the best checkpoint
MAX_PER_CLASS = 3000          # cap very large classes (e.g. segmented neutrophils) to speed training
MODEL_NAME  = "efficientnet_b0"
IMG_SIZE    = 224
BATCH_SIZE  = 64
EPOCHS      = 15
LR          = 3e-4
SEED        = 42
OUT_DIR     = "/content/haem5_model"
if QUICK:
    KAGGLE_DATASETS = [k for k in KAGGLE_DATASETS if "bone-marrow" not in k]
    MAX_PER_CLASS, EPOCHS = 1500, 8
    print("⚡ QUICK mode: blood datasets only, ≤1500 images/class, 8 epochs")

# --- Auto-detect Kaggle: datasets attached with "Add Input" are used directly (no zip, no Drive) ---
import os, glob as _g
import sys as _sys
IN_COLAB = "google.colab" in _sys.modules or "COLAB_RELEASE_TAG" in os.environ
ON_KAGGLE = (not IN_COLAB) and (bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/working"))
if ON_KAGGLE:
    DATASETS = [(d, os.path.basename(d.rstrip("/"))) for d in sorted(_g.glob("/kaggle/input/*/")) + sorted(_g.glob("/kaggle/input/*/*/"))
                if any(os.scandir(d))]
    # keep only top-level dataset folders (avoid double counting nested paths)
    tops = []
    for d, n in DATASETS:
        if not any(d.startswith(t) for t, _ in tops): tops.append((d, n))
    DATASETS = tops
    OUT_DIR = "/kaggle/working/haem5_model"
print("Running on", "Kaggle" if ON_KAGGLE else "Colab", "| datasets:", [n for _, n in DATASETS])

In [ ]:
# 3. Imports
import os, re, glob, zipfile, hashlib, json, time, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score,
                             classification_report, confusion_matrix)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| classes:", len(CLASSES))
if device.type == "cpu":
    # No GPU available: switch to a lighter model and a smaller sample so training still finishes (about 1–2 h).
    MODEL_NAME, IMG_SIZE, BATCH_SIZE, EPOCHS, MAX_PER_CLASS = "mobilenetv3_large_100", 160, 32, 6, 700
    torch.set_num_threads(os.cpu_count() or 2)
    print(f"⚠ No GPU — CPU mode: {MODEL_NAME}, {IMG_SIZE}px, {EPOCHS} epochs, ≤{MAX_PER_CLASS} images/class. Results will be somewhat lower than with a GPU.")

In [ ]:
# 4. Transforms — the app reproduces eval_tf exactly: resize to 224x224, RGB, ImageNet mean/std
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(180), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    transforms.ColorJitter(0.15, 0.15, 0.10, 0.02),   # stain / scanner variation
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

In [ ]:
# 5. Load datasets, map labels, remove duplicates, split
def kaggle_download(slug):
    import kagglehub
    try:
        return kagglehub.dataset_download(slug)
    except Exception as first:
        if ON_KAGGLE:
            print(f"  ⚠ Could not fetch {slug} ({type(first).__name__}). Attach it with + Add Input, or ignore it."); return None
        # Fall back to your Kaggle API key: Colab Secrets (key icon) KAGGLE_USERNAME / KAGGLE_KEY, or upload kaggle.json
        try:
            from google.colab import userdata
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME"); os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        except Exception:
            tok = input("Kaggle needs your API token. Paste it here and press Enter (or just press Enter to upload kaggle.json): ").strip()
            if tok:
                os.environ["KAGGLE_API_TOKEN"] = tok          # new-style token (Kaggle › Settings › API › Create New Token)
                if ":" in tok: os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = tok.split(":", 1)
                uname = input("Your Kaggle username (shown on your Kaggle profile): ").strip()
                if uname: os.environ["KAGGLE_USERNAME"] = uname; os.environ.setdefault("KAGGLE_KEY", tok)
            else:
                from google.colab import files
                up = files.upload(); kj = json.loads(next(iter(up.values())))
                os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = kj["username"], kj["key"]
        return kagglehub.dataset_download(slug)

if USE_KAGGLE_DOWNLOAD:
    have = {n.lower() for _, n in DATASETS}
    for slug in KAGGLE_DATASETS:
        name = slug.split("/")[-1]
        if name.lower() in have:
            print("Already attached:", name); continue
        print("Downloading", slug, "from Kaggle (a few minutes)…")
        path = kaggle_download(slug)
        if path:
            print("  →", path); DATASETS = DATASETS + [(path, name)]
if not ON_KAGGLE and any(p.startswith("/content/drive") for p, _ in DATASETS):
    from google.colab import drive
    drive.mount("/content/drive")

IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
def code_to_class(folder, fname):
    keys = []
    m = re.match(r"([A-Za-z]+)", os.path.basename(fname))
    if m: keys.append(m.group(1).upper())          # filename prefix first (e.g. PMY_/MY_/MMY_ inside PBC's "ig" folder)
    keys.append(os.path.basename(folder).upper())
    for k in keys:
        if k in SKIP_CODES: return None
        if k in LABEL_MAP: return LABEL_MAP[k]
    # fuzzy folder words (e.g. 'training data eosinophil')
    f = os.path.basename(folder).lower()
    for word, cls in [("metamyelo", "metamyelocyte"), ("promyelo", "promyelocyte"), ("myelocyte", "myelocyte"),
                      ("baso", "basophil"), ("eosin", "eosinophil"), ("lympho", "lymphocyte"), ("mono", "monocyte"),
                      ("erythroblast", "erythroblast"), ("platelet", "platelet")]:
        if word in f: return cls
    return None

rows, unmapped = [], {}
for zpath, name in DATASETS:
    if not os.path.exists(zpath):
        print(f"⚠ not found, skipping: {zpath}"); continue
    if os.path.isdir(zpath):
        dest = zpath                      # Kaggle input folder: read in place
    else:
      dest = f"/content/data/{name}"
      if not os.path.isdir(dest):
        print("Unzipping", zpath, "…")
        with zipfile.ZipFile(zpath) as z: z.extractall(dest)
    for p in glob.glob(f"{dest}/**/*", recursive=True):
        if not p.lower().endswith(IMG_EXT) or "__MACOSX" in p: continue
        cls = code_to_class(os.path.dirname(p), p)
        if cls is None:
            k = os.path.basename(os.path.dirname(p)); unmapped[k] = unmapped.get(k, 0) + 1; continue
        rows.append(dict(path=p, cls=cls, source=name))
df = pd.DataFrame(rows)
assert len(df), "No images mapped — check DATASETS paths and LABEL_MAP"
# Keep a generous random sample per class *before* hashing, so huge sets (171k marrow images) stay fast.
df = pd.concat([g.sample(min(len(g), int(MAX_PER_CLASS * 1.3)), random_state=SEED) for _, g in df.groupby("cls")]).reset_index(drop=True)
if unmapped: print("Unmapped folders (ignored):", unmapped)

df["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in df.path]
n0 = len(df); df = df.drop_duplicates("md5"); print(f"Removed {n0-len(df)} exact duplicate images")
conflict = df.groupby("md5").cls.nunique(); assert (conflict <= 1).all()

df = df[df.cls.isin(CLASSES)]
df = pd.concat([g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED) for _, g in df.groupby("cls")]).reset_index(drop=True)
present = [c for c in CLASSES if c in set(df.cls)]
missing = [c for c in CLASSES if c not in present]
if missing: print("⚠ No training images for:", missing, "— the model will never predict these.")

# ---- Split: 70% train / 15% validation / 15% test, stratified by class AND source dataset ----
# Exact duplicates were removed above, so no image can appear in two splits (no leakage).
strat = df.cls + "|" + df.source
rare = strat.map(strat.value_counts()) < 7          # groups too small to stratify jointly fall back to class only
strat = strat.where(~rare, df.cls)
tr, te = train_test_split(df, test_size=TEST_FRAC, stratify=strat, random_state=SEED)
tr, va = train_test_split(tr, test_size=VAL_FRAC / (1 - TEST_FRAC), stratify=strat.loc[tr.index], random_state=SEED)
assert not (set(tr.md5) & set(va.md5) or set(tr.md5) & set(te.md5) or set(va.md5) & set(te.md5))
pd.concat([tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")])[["path", "cls", "source", "md5", "split"]] \
  .to_csv(f"{OUT_DIR}/split_manifest.csv", index=False)
print(f"Split: train {len(tr)} ({len(tr)/len(df):.0%}) | val {len(va)} ({len(va)/len(df):.0%}) | test {len(te)} ({len(te)/len(df):.0%}) — saved split_manifest.csv")
c2i = {c: i for i, c in enumerate(CLASSES)}

class PathDS(Dataset):
    def __init__(s, d, tf): s.p, s.y, s.tf = list(d.path), [c2i[c] for c in d.cls], tf
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.tf(Image.open(s.p[i]).convert("RGB")), s.y[i]

train_ds, val_ds, test_ds = PathDS(tr, train_tf), PathDS(va, eval_tf), PathDS(te, eval_tf)
y_train = np.array(train_ds.y)
summary = pd.DataFrame({"train": tr.cls.value_counts(), "val": va.cls.value_counts(), "test": te.cls.value_counts()}).reindex(CLASSES).fillna(0).astype(int)
print(summary); print(pd.crosstab(df.cls, df.source))
kw = dict(batch_size=BATCH_SIZE, num_workers=min(4, os.cpu_count() or 2), pin_memory=True, persistent_workers=True)
train_dl = DataLoader(train_ds, shuffle=True, drop_last=True, **kw)
val_dl, test_dl = DataLoader(val_ds, **kw), DataLoader(test_ds, **kw)

In [ ]:
# 6. ✅ VERIFY THE LABEL MAPPING — 4 random images per class. Fix LABEL_MAP in cell 2 if anything is misplaced.
show = [c for c in CLASSES if c in set(df.cls)]
fig, axes = plt.subplots(len(show), 4, figsize=(9, 2.3 * len(show)))
for r, c in enumerate(show):
    sub = df[df.cls == c].sample(min(4, (df.cls == c).sum()), random_state=1)
    for k in range(4):
        ax = axes[r, k]; ax.axis("off")
        if k < len(sub):
            ax.imshow(Image.open(sub.path.iloc[k]).convert("RGB"))
            ax.set_title(f"{c}\n{os.path.basename(os.path.dirname(sub.path.iloc[k]))}", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 7. Model
try:
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(CLASSES)).to(device)
except Exception as e:
    raise SystemExit(f"❌ Could not download pretrained weights ({type(e).__name__}). Turn Internet ON and rerun from cell 1.")
counts = np.bincount(y_train, minlength=len(CLASSES)).astype(float)
w = np.where(counts > 0, counts.sum() / (len(present) * np.maximum(counts, 1)), 0.0)
w = np.minimum(w, 10.0)   # cap: very rare classes would otherwise get weights >100 and destabilise training
criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device), label_smoothing=0.05)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_dl), pct_start=0.15)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

In [ ]:
# 8. Train (best checkpoint by validation macro-F1)
@torch.no_grad()
def predict(dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            P.append(model(x.to(device)).float().softmax(1).cpu())
        Y.append(torch.as_tensor(y))
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

best, ckpt, hist = -1, f"{OUT_DIR}/best.pt", []
for ep in range(1, EPOCHS + 1):
    model.train(); t0, L, n = time.time(), 0., 0
    for x, y in train_dl:
        x, y = x.to(device), torch.as_tensor(y).to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        L += loss.item() * len(x); n += len(x)
    p, y = predict(val_dl); f1 = f1_score(y, p.argmax(1), average="macro", labels=[c2i[c] for c in present])
    hist.append((ep, L / n, f1)); flag = ""
    if f1 > best: best = f1; torch.save(model.state_dict(), ckpt); flag = " ✓"
    print(f"Ep {ep:02d} loss {L/n:.4f} val macro-F1 {f1:.4f} {time.time()-t0:.0f}s{flag}")

In [ ]:
# 9. Test-set evaluation (incl. blast detection, the clinically critical class)
model.load_state_dict(torch.load(ckpt, map_location=device))
tp, ty = predict(test_dl); pred = tp.argmax(1)
labs = [c2i[c] for c in present]
print(f"Accuracy {accuracy_score(ty, pred):.4f} | balanced {balanced_accuracy_score(ty, pred):.4f} | "
      f"macro-F1 {f1_score(ty, pred, average='macro', labels=labs):.4f} | kappa {cohen_kappa_score(ty, pred):.4f}")
print(classification_report(ty, pred, labels=labs, target_names=present, digits=3, zero_division=0))
cm = confusion_matrix(ty, pred, labels=labs); cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
plt.figure(figsize=(11, 9)); sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", xticklabels=present, yticklabels=present, vmin=0, vmax=1)
plt.title("Recall-normalised confusion matrix"); plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=200); plt.show()

if "blast" in present:
    b = c2i["blast"]; tb, pb = ty == b, pred == b
    sens = (tb & pb).sum() / max(tb.sum(), 1); spec = (~tb & ~pb).sum() / max((~tb).sum(), 1)
    ppv = (tb & pb).sum() / max(pb.sum(), 1)
    print(f"\nBLAST vs non-blast: sensitivity {sens:.3f}  specificity {spec:.3f}  PPV {ppv:.3f}  (n blasts = {tb.sum()})")
    print("Most frequent cells mis-called as blast:", pd.Series([CLASSES[t] for t in ty[pb & ~tb]]).value_counts().head(5).to_dict())

In [ ]:
# 9b. Blast threshold calibration — choose on VALIDATION, report on TEST. Missing a blast is worse than flagging one.
vp, vy = predict(val_dl)
BLAST_THR = None
if "blast" in present:
    b = c2i["blast"]
    def blast_stats(P, Y, t):
        pred = P.argmax(1).copy(); pred[P[:, b] >= t] = b
        tb, pb = Y == b, pred == b; tpos = (tb & pb).sum()
        sens, ppv = tpos / max(tb.sum(), 1), tpos / max(pb.sum(), 1)
        return sens, ppv, 5 * ppv * sens / max(4 * ppv + sens, 1e-9)
    tab = pd.DataFrame([(t, *blast_stats(vp, vy, t)) for t in np.round(np.arange(0.10, 0.51, 0.05), 2)],
                       columns=["threshold", "sensitivity", "PPV", "F2"]).round(3)
    print(tab)
    ok = tab[tab.PPV >= 0.70]
    BLAST_THR = float(ok.sort_values("F2").iloc[-1].threshold) if len(ok) else None
    if BLAST_THR:
        s_, p_, _ = blast_stats(tp, ty, BLAST_THR)
        print(f"\nBlast threshold {BLAST_THR}: TEST sensitivity {s_:.3f}, PPV {p_:.3f}")

In [ ]:
# 10. Export to ONNX for the app, verify against PyTorch, download
import onnxruntime as ort
model.eval().cpu()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = f"{OUT_DIR}/cell_model.onnx"
torch.onnx.export(model, dummy, onnx_path, input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)

xb = torch.stack([test_ds[i][0] for i in range(min(16, len(test_ds)))])
with torch.no_grad(): ref = model(xb).numpy()
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
out = sess.run(None, {"input": xb.numpy()})[0]
diff = np.abs(out - ref).max(); print(f"ONNX vs PyTorch max |Δlogit| = {diff:.2e}", "✓" if diff < 1e-3 else "⚠ check export")
assert (out.argmax(1) == ref.argmax(1)).all()

meta = {"format": "haem5-cell-v1", "classes": CLASSES, "present_classes": present, "img_size": IMG_SIZE,
        "mean": MEAN, "std": STD, "input_name": "input", "layout": "NCHW", "resize": "direct",
        "model": MODEL_NAME, "trained_on": [n for p, n in DATASETS if os.path.exists(p)],
        "n_train": int(len(train_ds)), "thresholds": {"blast": BLAST_THR} if BLAST_THR else {},
        "test": {"accuracy": float(accuracy_score(ty, pred)), "macro_f1": float(f1_score(ty, pred, average='macro', labels=labs))}}
json.dump(meta, open(f"{OUT_DIR}/cell_labels.json", "w"), indent=2)
print(f"Model size: {os.path.getsize(onnx_path)/1e6:.1f} MB")

if ON_KAGGLE:
    print("Done. Download cell_model.onnx and cell_labels.json from the Output panel (right side) → haem5_model.")
else:
    from google.colab import files
    files.download(onnx_path); files.download(f"{OUT_DIR}/cell_labels.json")
# !cp -r /content/haem5_model /content/drive/MyDrive/   # keep a copy in Drive

## Load into the app
Open **HAEM5 Reporter › Differential counter › AI-assisted**, click *Load model*, select **both** `cell_model.onnx` and `cell_labels.json`. Then add single-cell crops (CellaVision exports, or crops you cut from a smear photo). The app classifies each cell, lets you correct any label, and adds the tally to the differential that feeds the WHO-HAEM5 engine.

**Validation before any real use**
- Report blast sensitivity/PPV from cell 9 — false blasts directly inflate WHO-HAEM5 categories (e.g. MDS-IB1 → IB2, MDS → AML).
- Validate on local smears (Leishman/MGG, your microscope and camera) against a 200/500-cell manual differential by two observers; report agreement (ICC/Bland–Altman for blast %).
- Promonocytes are blast equivalents in WHO-HAEM5 but have no class in these datasets — count them manually.